<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/mcp-arxiv-hero.svg" align="center" width="35%">
</div>

<br>

# BUILDING MCP CLIENTS

<br>

**About:** This notebook builds the client side of tool-calling: the loop that sends messages to the model, routes responses by content type, executes tools when the model asks, and threads results back into the conversation.

**Learning Goals:** Initialize an LLM client with tool support, route text and tool_use responses correctly, manage conversation state across multiple turns, and build an interactive query loop that terminates cleanly.

**Keywords:** mcp, client, tool-calling, message-flow, anthropic-api, conversation-state

**Prerequisite Knowledge:** (1) Completed [Building MCP Servers](01_building_mcp_servers.ipynb) or comfortable with tool schemas and dispatchers, (2) Python basics (functions, classes, control flow), (3) an Anthropic API key (only for the live examples)

**Target User:** Developers integrating LLM tool-calling into a chatbot, agent, or automation, and who need to reason about the shape of the message loop.


<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>


#### CONTENTS

> #### [PART 0: THE CLIENT-SERVER MODEL](#Part_0)
> #### [PART 1: INITIALIZING A CLIENT](#Part_1)
> #### [PART 2: SENDING MESSAGES AND TOOLS](#Part_2)
> #### [PART 3: ROUTING RESPONSES](#Part_3)
> #### [PART 4: MANAGING STATE AND BUILDING THE LOOP](#Part_4)

#### APPENDIX

> #### [COMPANION MATERIALS](#Appendix_1)
> #### [REFERENCES AND FURTHER READING](#Appendix_2)

<br>


<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **THE** CLIENT-SERVER **MODEL**


<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/response-routing.svg" align="center" width="60%" padding="10"><br>
    <br>
    A single model response can contain both text (user-facing) and tool_use (needs dispatch). The client decides how each block flows.
</div>

<br>

Notebook 01 built the **server**: the tools, their schemas, and the dispatcher that maps a name-plus-arguments request to a Python call. Notebook 02 builds the **client**: the loop that runs on your side of the API and drives the whole interaction.

The client owns five responsibilities:

1. Accept the user's query.
2. Send it to the model along with the current tool schemas.
3. Interpret the response - which is a *list* of content blocks, some `text` and some `tool_use`.
4. When the model returns `tool_use`, call the dispatcher, then post the result back into the conversation.
5. Loop until the model returns a response with no `tool_use` blocks - that terminal response is the answer for the user.

This split matters because it is what lets you swap the *presentation* (a terminal loop, a Slack bot, a web endpoint) without touching the *tools*. Every one of those front ends calls the same `process_query`.


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **INITIALIZING** A **CLIENT**


#### **1.1 Loading the API key**
___

Never hard-code an API key. Store it in the environment (a `.env` file loaded by `python-dotenv` is the low-friction default) so it stays out of git and out of shared notebooks.


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    print("ANTHROPIC_API_KEY not found. Live examples in this notebook will be skipped.")
else:
    print("API key loaded.")


Failing early is deliberate. If we let the code continue with `api_key = None`, the first API call would return a cryptic auth error many cells later. Catching the missing key here gives an immediate, actionable message.

___

**Note:** `python-dotenv` reads a `.env` file placed next to the notebook. Add `.env` to your `.gitignore` before you commit anything.

___


#### **1.2 Creating the client**
___


In [ ]:
if api_key:
    client = anthropic.Anthropic(api_key=api_key)
    print(f"Client ready: {type(client).__name__}")
else:
    client = None
    print("No client (missing API key). Downstream cells will noop.")


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **SENDING** MESSAGES AND **TOOLS**


Every call to `client.messages.create` needs two things beyond the model identifier:

- **`messages`** - the conversation so far. The API is stateless; if you do not send the history the model does not have it.
- **`tools`** - the list of schemas we built in Notebook 01. If you omit this the model has no tools to call.

Both are passed by value on every request.


#### **2.1 Redeclaring the schemas locally**
___

In a real application you would `from server import tools` (or import from wherever your schemas live). We redeclare inline so this notebook is self-contained.


In [ ]:
tools = [
    {
        "name": "search_papers",
        "description": "Search arXiv for academic papers on a topic. Returns a list of paper IDs.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {"type": "string", "description": "The topic to search for."},
                "max_results": {"type": "integer", "description": "Maximum results.", "default": 5},
            },
            "required": ["topic"],
        },
    },
    {
        "name": "extract_info",
        "description": "Retrieve stored metadata for one paper by its arXiv short ID.",
        "input_schema": {
            "type": "object",
            "properties": {
                "paper_id": {"type": "string", "description": "arXiv short ID from search_papers."},
            },
            "required": ["paper_id"],
        },
    },
]
print(f"{len(tools)} tool schemas registered")


#### **2.2 The shape of a request**
___


In [ ]:
user_query = "Find papers about retrieval augmented generation."

messages = [{"role": "user", "content": user_query}]

# The API call itself:
#   response = client.messages.create(
#       model="claude-sonnet-4-6",
#       max_tokens=2048,
#       tools=tools,
#       messages=messages,
#   )
# We do not run it here to keep the notebook offline. See section 4.2 for the live version.
print("Messages:", messages)
print("Tools registered:", [t["name"] for t in tools])


The response you get back has a `.content` attribute that is a *list*. That list is the crux of routing - every element is a block with a `.type` field, and your loop dispatches on that type.

___

**Note:** Model identifiers are volatile. Verify the current model list at [docs.anthropic.com/en/docs/about-claude/models](https://docs.anthropic.com/en/docs/about-claude/models) before pinning `claude-sonnet-4-6` (or any specific version) in production code.

___


<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->


> **Question:** The API is stateless: you must send the entire `messages` list on every call. In two sentences, explain what this design choice buys you (and what it costs).

<br>

```python
# Draft your answer as a Python string.
answer = "..."
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **ROUTING** RESPONSES


Every response.content block is one of these types (the two you will handle):

- **`text`** - free-form model output. This is what the user sees.
- **`tool_use`** - a request to call a tool. The block has three fields you care about: `.name`, `.input`, and `.id`. The `id` is how you correlate the eventual `tool_result` message back to the request.

A single response can mix them: the model may explain what it is about to do (`text`) and then request the tool (`tool_use`) in the same list. Route each block independently.


#### **3.1 Mock response objects**
___

We build tiny shims that mimic the real SDK objects so the routing code runs without an API call.


In [ ]:
from dataclasses import dataclass


@dataclass
class MockText:
    text: str
    type: str = "text"


@dataclass
class MockToolUse:
    id: str
    name: str
    input: dict
    type: str = "tool_use"


@dataclass
class MockResponse:
    content: list


#### **3.2 The routing pattern**
___


In [ ]:
def handle_response(response) -> list:
    """
    Walk each content block and dispatch by type.

    Returns:
        List of tuples describing what should happen next:
          ("show", text_str)  - the user should see this text
          ("call", tool_name, tool_args, tool_id) - the dispatcher should be invoked
    """
    actions = []
    for block in response.content:
        if block.type == "text":
            actions.append(("show", block.text))
        elif block.type == "tool_use":
            actions.append(("call", block.name, block.input, block.id))
        else:
            # Anthropic may add block types over time. Fail loudly during dev,
            # log-and-continue in production.
            raise ValueError(f"Unknown block type: {block.type!r}")
    return actions


mixed = MockResponse(content=[
    MockText("Let me search for that."),
    MockToolUse(id="toolu_1", name="search_papers", input={"topic": "rag"}),
])

for action in handle_response(mixed):
    print(action)


The dispatch table above is the single most important piece of client code. Every other piece of the loop is bookkeeping around it.

The `raise ValueError` on unknown block types is a deliberate choice for tutorial code: better to fail loudly and learn about a new block type than to silently drop it. In production you would log-and-continue instead so the conversation does not die.


<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->


> **Question:** Write a function `classify_response(response)` that returns `"text_only"`, `"tool_only"`, or `"mixed"` based on the content block types present. This classification is what the outer loop uses to decide whether to terminate.

<br>

```python
def classify_response(response) -> str:
    ### YOUR CODE HERE ###
    ...
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **MANAGING** STATE AND **BUILDING** THE LOOP


Conversation state is the `messages` list. Every exchange - user query, assistant response, tool result - appends to it. Every subsequent API call sends the entire list back.

There are three message shapes you will produce:

- **User turn** - `{"role": "user", "content": "<plain text>"}`.
- **Assistant response** - `{"role": "assistant", "content": response.content}` (list of blocks straight from the API).
- **Tool result** - `{"role": "user", "content": [{"type": "tool_result", "tool_use_id": "...", "content": "<string>"}]}`.

The counterintuitive one is the third: tool results have `role="user"` even though *you* produced them, because the API treats tool output as "information coming back from outside the model's own reasoning" - the same category as a user typing a message.


#### **4.1 Building a tool_result message**
___


In [ ]:
def build_tool_result_message(tool_use_block, result: str) -> dict:
    """
    Package a dispatcher return value as the message the API expects.

    Note the shape: role='user', content is a *list* containing one dict.
    The tool_use_id must match the id from the tool_use block, so the model can
    thread the result back to its own request.
    """
    return {
        "role": "user",
        "content": [{
            "type": "tool_result",
            "tool_use_id": tool_use_block.id,
            "content": result,
        }],
    }


sample_block = MockToolUse(id="toolu_abc", name="search_papers", input={"topic": "rag"})
sample_result = "2501.11111, 2501.22222, 2501.33333"

print(build_tool_result_message(sample_block, sample_result))


#### **4.2 The full `process_query` loop**
___

Below is the shape of the loop as it appears in `arxiv_chatbot.py`. This code will only *run* if you have a live client and the tools from Notebook 01 in scope, but you can read and trace it either way.


In [ ]:
def process_query(query: str, client, tools, execute_tool_fn) -> None:
    """
    Drive one user query through the model with tool support.

    Args:
        query: The user's natural-language question.
        client: An initialized anthropic.Anthropic client.
        tools: The list of tool schemas (from Notebook 01).
        execute_tool_fn: Callable (name, args) -> str; the dispatcher from
            Notebook 01, Part 3.
    """
    messages = [{"role": "user", "content": query}]

    response = client.messages.create(
        model="claude-3-7-sonnet-20250219",
        max_tokens=2048,
        tools=tools,
        messages=messages,
    )

    while True:
        assistant_blocks = list(response.content)
        made_tool_call = False

        for block in response.content:
            if block.type == "text":
                print(block.text)
            elif block.type == "tool_use":
                made_tool_call = True
                messages.append({"role": "assistant", "content": assistant_blocks})

                tool_result = execute_tool_fn(block.name, block.input)
                messages.append({
                    "role": "user",
                    "content": [{
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": tool_result,
                    }],
                })

                response = client.messages.create(
                    model="claude-3-7-sonnet-20250219",
                    max_tokens=2048,
                    tools=tools,
                    messages=messages,
                )
                break  # re-enter the outer loop with the new response

        if not made_tool_call:
            return  # response was text-only; conversation is done


print("process_query defined. See arxiv_chatbot.py for the runnable version.")


The loop terminates when a response arrives with no `tool_use` blocks. Every intermediate response may add one or more tool calls; each triggers another API round-trip. That is why a single user query can produce three or four API calls.


#### **4.3 The interactive shell**
___


In [ ]:
def chat_loop(client, tools, execute_tool_fn) -> None:
    """
    Run an interactive REPL that dispatches each query through process_query.

    Ctrl-C exits cleanly; other exceptions are surfaced but do not tear down
    the loop - the user can retry after a bad query.
    """
    print("Type your queries or 'quit' to exit.\n")
    while True:
        try:
            query = input("Query: ").strip()
            if query.lower() == "quit":
                break
            process_query(query, client, tools, execute_tool_fn)
            print()
        except KeyboardInterrupt:
            print("\nInterrupted. Goodbye.")
            break
        except Exception as e:
            print(f"Error: {e}")


print("chat_loop defined. Run arxiv_chatbot.py to try it live.")


The `except Exception` swallow is intentional here. In an interactive shell, a single bad query should not force the user to restart the process; log the error and let them try again. A batch application would make the opposite choice - let exceptions propagate so the run fails loudly.


<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->


> **Question:** The `process_query` loop above has no upper bound on iteration count. Under what circumstances would that be a problem, and how would you add a safety rail? Write a modified loop that caps at 5 tool round-trips.

<br>

```python
def process_query_bounded(query, client, tools, execute_tool_fn, max_calls=5):
    ### YOUR CODE HERE ###
    ...
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Appendix_1'></a>

<hr style="border: 2px solid#003262;" />

#### APPENDIX

## **COMPANION** MATERIALS


- **Homework**: [Building MCP Clients Homework](homework/building_mcp_clients_homework.ipynb) - practice response classification, message construction, and loop termination on mocked responses (no API key required).
- **Working script**: `arxiv_chatbot.py` - the full server + client implementation this notebook walks through.
- **Previous notebook**: [Building MCP Servers](01_building_mcp_servers.ipynb) - the tools and dispatcher this client drives.
- **Next notebook**: [MCP in Practice](03_mcp_in_practice.ipynb) - end-to-end walkthrough, tool design principles, and deployment patterns.


<a id='Appendix_2'></a>

<hr style="border: 2px solid#003262;" />

##### **REFERENCES**


- [Anthropic Messages API - Tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [Anthropic tool_use and tool_result content blocks](https://docs.anthropic.com/en/docs/build-with-claude/tool-use#tool-use-and-tool-result-content-blocks)
- [Model Context Protocol specification](https://modelcontextprotocol.io/)
- [python-dotenv documentation](https://pypi.org/project/python-dotenv/)


<hr style="border: 6px solid#003262;" />
